# 🧪 PostgreSQL Connection & API Diagnostic Notebook

This notebook verifies your environment, database connectivity, and backend API endpoints without exposing sensitive credentials.

## Step 1: Import Dependencies

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.engine import URL
from dotenv import load_dotenv
import os
import requests

print("✅ Core packages imported successfully!")

## Step 2: Load Environment Variables from `backend/.env`

In [ ]:
# Load .env from backend directory
env_path = Path("backend") / ".env"
load_dotenv(dotenv_path=env_path, override=True)

db_env = os.getenv("DB_ENV", "remote").lower().strip()
schema_name = os.getenv("METADATA_SCHEMA", "public")
table_name = os.getenv("METADATA_TABLE", "tts_metadata")

if db_env == "local":
    db_host = os.getenv("LOCAL_DB_HOST", "localhost")
    db_port = int(os.getenv("LOCAL_DB_PORT", "5432"))
    db_name = os.getenv("LOCAL_DB_NAME", "test_tts")
    db_user = os.getenv("LOCAL_DB_USER", "postgres")
    db_password = os.getenv("LOCAL_DB_PASSWORD", "docker123")
else:
    db_host = os.getenv("DB_HOST", "localhost")
    db_port = int(os.getenv("DB_PORT", "5432"))
    db_name = os.getenv("DB_NAME")
    db_user = os.getenv("DB_USER")
    db_password = os.getenv("DB_PASSWORD")

print(f"Mode:          {db_env}")
print(f"Target Host:   {db_host}:{db_port}")
print(f"Target DB:     {db_name}")
print(f"Target User:   {db_user}")
print(f"Target Schema: {schema_name}")
print(f"Target Table:  {table_name}")

## Step 3: Test PostgreSQL Connectivity

In [ ]:
# Construct database URL
db_url = URL.create(
    drivername="postgresql+psycopg",
    username=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    database=db_name,
)

engine = create_engine(db_url, pool_pre_ping=True)

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version()")).scalar()
        print("✅ PostgreSQL connection successful!")
        print(f"Version: {result}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

## Step 4: Verify Table Existence and Inspect Columns

In [ ]:
try:
    inspector = inspect(engine)
    tables = inspector.get_table_names(schema=schema_name)
    
    if table_name in tables:
        print(f"✅ Table '{schema_name}.{table_name}' exists!")
        columns = inspector.get_columns(table_name, schema=schema_name)
        col_names = [col['name'] for col in columns]
        print(f"Columns ({len(col_names)}): {col_names}")
    else:
        print(f"❌ Table '{table_name}' NOT found in schema '{schema_name}'.")
        print(f"Available tables: {tables}")
except Exception as e:
    print(f"❌ Inspection failed: {e}")

## Step 5: Check Record Count & Preview Top 5 Records

In [ ]:
try:
    with engine.connect() as conn:
        count_query = text(f"SELECT COUNT(*) FROM {schema_name}.{table_name}")
        total = conn.execute(count_query).scalar()
        print(f"📊 Total records in '{table_name}': {total}")
        
        preview_query = text(f"SELECT * FROM {schema_name}.{table_name} LIMIT 5")
        df_preview = pd.read_sql_query(preview_query, conn)
        
    display(df_preview)
except Exception as e:
    print(f"❌ Query failed: {e}")

## Step 6: Test FastAPI Backend Endpoints (When FastAPI is Running)

In [ ]:
api_base = "http://127.0.0.1:8000"

try:
    health_resp = requests.get(f"{api_base}/health", timeout=3)
    print("Health Endpoint:", health_resp.json())
    
    db_health = requests.get(f"{api_base}/health/database", timeout=3)
    print("DB Health Endpoint:", db_health.json())
    
    count_resp = requests.get(f"{api_base}/api/v1/metadata/count", timeout=3)
    print("Metadata Count Endpoint:", count_resp.json())
    
    metadata_resp = requests.get(f"{api_base}/api/v1/metadata?limit=3&offset=0", timeout=3)
    print("Metadata Page Sample:", metadata_resp.json())
except requests.exceptions.ConnectionError:
    print("⚠️ FastAPI is not currently running at http://127.0.0.1:8000. Start it using uvicorn!")
except Exception as e:
    print(f"Error: {e}")